In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Data processing

In [ ]:
# Load data
train = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
test = pd.read_csv('/kaggle/input/digit-recognizer/test.csv')

# Scale pixel values from [0, 255] to [0, 1]
X_train = train.drop("label", axis=1).values.astype(np.float32) / 255.0
y_train = train["label"].values.astype(np.int64)
X_test = test.values.astype(np.float32) / 255.0

# Reshape to (N, C, H, W) format for PyTorch CNNs
X_train = X_train.reshape(-1, 1, 28, 28)
X_test = X_test.reshape(-1, 1, 28, 28)


# Split training data into train (90%) and validation (10%) sets with stratification
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42, stratify=y_train
)

print(f"Train: {X_train_split.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

In [ ]:
class DigitDataset(Dataset):
    """Custom PyTorch Dataset for loading digit images with optional transforms"""
    def __init__(self, X, y, transform=None):
        self.X = X if isinstance(X, np.ndarray) else X.numpy()
        self.y = y if isinstance(y, np.ndarray) else y.numpy()
        self.transform = transform
   
    def __len__(self):
        return len(self.X)
   
    def __getitem__(self, idx):
        # Convert to tensor
        image = torch.FloatTensor(self.X[idx])
        label = torch.LongTensor([self.y[idx]])[0]
       
        # Apply augmentation (if any)
        if self.transform:
            image = self.transform(image)
       
        return image, label


# MNIST standard statistics (more accurate than calculating from data)
mnist_mean = 0.1307
mnist_std = 0.3081

# Training transforms: Augmentation + Normalization
train_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.1, 0.1),
        scale=(0.9, 1.1),
        shear=5
    ),
    # Normalize AFTER augmentation
    transforms.Normalize(mean=[mnist_mean], std=[mnist_std])
])

# Validation/Test transforms: Only normalization, NO augmentation
val_transform = transforms.Compose([
    transforms.Normalize(mean=[mnist_mean], std=[mnist_std])
])

# Create datasets
train_dataset = DigitDataset(X_train_split, y_train_split, transform=train_transform)
val_dataset = DigitDataset(X_val, y_val, transform=val_transform)
test_dataset = DigitDataset(X_test, np.zeros(len(X_test), dtype=np.int64), transform=val_transform)

# DataLoaders with optimized settings
train_loader = DataLoader(
    train_dataset, 
    batch_size=128,     
    shuffle=True,
    num_workers=2,       # Parallel data loading
    pin_memory=True      # Faster GPU transfer
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=256,      
    shuffle=False,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=256,
    shuffle=False,
    pin_memory=True
)

print(f"\nDataLoader Info:")
print(f"Train batches: {len(train_loader)} (batch_size=128)")
print(f"Val batches: {len(val_loader)} (batch_size=256)")
print(f"Test batches: {len(test_loader)} (batch_size=256)")

# CNN Model

In [ ]:
class DigitCNN(nn.Module):
    """
    CNN Architecture for digit classification with two convolutional blocks.
    
    Key Techniques:
    - BatchNormalization: Stabilizes training, allows higher learning rates
    - Dropout: Prevents overfitting by randomly dropping neurons during training
    - MaxPooling: Reduces spatial dimensions, extracts dominant features
    """
    def __init__(self):
        super().__init__()
        
        # Feature Extraction: Two convolutional blocks
        # Block 1: 1→32 channels, maintains 28x28 spatial size with padding
        # Block 2: 32→64 channels, reduces to 14x14 after MaxPool
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(32),
            nn.MaxPool2d(2), nn.Dropout(0.5),  # 28x28 → 14x14

            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(64),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm2d(64),
            nn.MaxPool2d(2), nn.Dropout(0.5)  # 14x14 → 7x7
        )
        
        # Classifier: Dense layers for final prediction
        # 64 channels × 7×7 spatial = 3136 features → 256 → 10 classes
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.6),
            nn.Linear(256, 10)  # 10 output logits (one per digit 0-9)
        )
    
    def forward(self, x):
        return self.classifier(self.features(x))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DigitCNN().to(device)
print(f"Model on: {device}")

**Loss function, optimizer, scheduler**

In [ ]:
num_epochs = 50

# Loss Function with Label Smoothing (0.1)
# Prevents overconfident predictions, improves generalization
# Instead of hard targets [0,0,1,0,...], uses soft targets [0.01, 0.01, 0.91, 0.01,...]
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

# AdamW Optimizer: Adam with decoupled weight decay
# Better regularization than L2 penalty in standard Adam
optimizer = optim.AdamW(
    model.parameters(),
    lr=0.001,              # Initial learning rate
    weight_decay=0.002,     # L2 regularization strength
    betas=(0.9, 0.999),    # Exponential decay rates for momentum
    eps=1e-8               # Numerical stability constant
)

# Cosine Annealing Learning Rate Scheduler
# Smoothly decreases LR from 0.001 → 1e-6 following cosine curve
# Better convergence than step decay or plateau-based schedulers
scheduler = CosineAnnealingLR(
    optimizer,
    T_max=num_epochs,      # Complete one cosine cycle over all epochs
    eta_min=5e-6           # Minimum learning rate at the end
)

# Train function

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """
    Train model for one epoch.
    Returns: (average_loss, accuracy_percentage)
    """
    model.train()  # Enable dropout and batch norm training mode
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc='Training', leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()  # Clear gradients from previous step
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimization
        loss.backward()  # Compute gradients
        optimizer.step()  # Update weights
        
        # Track metrics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({'loss': loss.item(), 'acc': 100. * correct / total})
    
    return running_loss / len(loader), 100. * correct / total

def validate(model, loader, criterion, device):
    """
    Evaluate model on validation set.
    Returns: (average_loss, accuracy_percentage)
    """
    model.eval()  # Disable dropout, use batch norm running stats
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc='Validation', leave=False)
    with torch.no_grad():  # Disable gradient computation for efficiency
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            pbar.set_postfix({'loss': loss.item(), 'acc': 100. * correct / total})
    
    return running_loss / len(loader), 100. * correct / total

# Training model

In [ ]:
"""
Training Loop with Multiple Advanced Techniques:

1. Early Stopping: Stops training if validation loss doesn't improve for `patience` epochs
   - Prevents overfitting by stopping before model memorizes training data
   
2. Cosine Annealing: Scheduler automatically reduces LR each epoch
   - Smooth LR decay helps fine-tune weights in later epochs
   
3. Best Model Tracking: Keeps track of best validation performance
   - Ensures we can identify the optimal model state
"""

best_val_loss = float('inf')
best_val_acc = 0.0
patience = 15  # Higher patience since cosine annealing naturally adjusts LR
patience_counter = 0

# History tracking for visualization
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [],
    'lr': []
}

for epoch in range(num_epochs):
    current_lr = optimizer.param_groups[0]['lr']
    print(f"\nEpoch {epoch+1}/{num_epochs} | LR: {current_lr:.6f}")
    print("-" * 60)
    
    # Train and validate
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    # Record metrics
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)
    
    # Update learning rate (cosine schedule)
    scheduler.step()
    
    print(f"Train → Loss: {train_loss:.4f} | Acc: {train_acc:.2f}%")
    print(f"Val   → Loss: {val_loss:.4f} | Acc: {val_acc:.2f}%")
    
    # Early stopping logic
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_acc = val_acc
        print("✓ New best model")
        patience_counter = 0  # Reset counter
    else:
        patience_counter += 1
        print(f"⚠ Patience: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print(f"\n🛑 Early stopping at epoch {epoch+1}")
            break

print(f"\n✓ Training complete. Best Val Acc: {best_val_acc:.2f}%")

# Learning curve

In [ ]:
"""
Visualize training dynamics: Loss and Accuracy curves over epochs.
Helps identify overfitting (gap between train/val) and learning progression.
"""

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history['train_loss']) + 1)

ax1.plot(epochs_range, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
ax1.plot(epochs_range, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history['train_acc'], 'b-', label='Train Acc', linewidth=2)
ax2.plot(epochs_range, history['val_acc'], 'r-', label='Val Acc', linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"Best validation accuracy: {max(history['val_acc']):.2f}%")
print(f"Final learning rate: {history['lr'][-1]:.6f}")

# Evaluation

In [ ]:
model.eval()
predictions = []

with torch.no_grad():  # Disable gradient computation for inference
    for images, _ in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)  # Get class with highest probability
        predictions.extend(predicted.cpu().numpy())

# Create submission file
submission = pd.DataFrame({
    "ImageId": list(range(1, len(predictions) + 1)),
    "Label": predictions
})
submission.to_csv("submission.csv", index=False)
print(f"✓ Submission saved: {len(predictions)} predictions")